In [1]:
import laspy
import numpy as np
import rasterio
from rasterio.transform import rowcol
from pyproj import Transformer
from pathlib import Path

# ============================================================
# INPUT
# ============================================================
GRID_PATH = "/home/b085164/miniconda3/envs/limatch/lib/python3.9/site-packages/pyproj/proj_dir/share/proj/CH/ch_swisstopo_chgeo2004_ETRS89_LN02.tif"
in_dir    = Path("/home/b085164/PDM_Romain_Defferrard/ECCR")
out_dir   = Path("/home/b085164/PDM_Romain_Defferrard/ECCR/LV95")

out_dir.mkdir(parents=True, exist_ok=True)

CHUNK = 1_000_000

lv95_to_etrs = Transformer.from_crs("EPSG:2056", "EPSG:4258", always_xy=True)

files = sorted(in_dir.glob("*.laz")) + sorted(in_dir.glob("*.las"))

with rasterio.open(GRID_PATH) as src:
    grid = src.read(1)

    for laz_path in files:
        out_path = out_dir / laz_path.name
        print(f"\n{'='*60}")
        print(laz_path.name)

        with laspy.open(laz_path) as reader:
            header = reader.header

            with laspy.open(out_path, mode="w", header=header) as writer:

                for points in reader.chunk_iterator(CHUNK):

                    x = points.x
                    y = points.y
                    z = points.z

                    # reprojection
                    lons, lats, _ = lv95_to_etrs.transform(x, y, z)

                    rows, cols = rowcol(src.transform, lons, lats)

                    # sécuriser indices
                    rows = np.clip(rows, 0, grid.shape[0] - 1)
                    cols = np.clip(cols, 0, grid.shape[1] - 1)

                    N = grid[rows, cols]

                    z_ellips = z + N

                    points.z = z_ellips

                    writer.write_points(points)

        print(f"-> {out_path}")

print("Terminé.")


ECCR_MN95_NF02_2533200-1155200.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533200-1155200.laz

ECCR_MN95_NF02_2533200-1155300.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533200-1155300.laz

ECCR_MN95_NF02_2533200-1155400.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533200-1155400.laz

ECCR_MN95_NF02_2533300-1154300.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533300-1154300.laz

ECCR_MN95_NF02_2533300-1154400.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533300-1154400.laz

ECCR_MN95_NF02_2533300-1154500.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533300-1154500.laz

ECCR_MN95_NF02_2533300-1154600.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533300-1154600.laz

ECCR_MN95_NF02_2533300-1154700.laz
-> /home/b085164/PDM_Romain_Defferrard/ECCR/LV95/ECCR_MN95_NF02_2533300-1154700.laz

ECCR_MN95_NF02_2533300-1154800.laz
-> /